# AdaBoost

In [1]:
import scipy.io
import numpy as np
import sklearn
from sklearn.feature_extraction import image
import matplotlib.pyplot as plt
from helper_functions import *

## Dataset - Data

In [2]:
data = scipy.io.loadmat('Data/data.mat')
faces = np.array(data['face'])
#number of classes 
M = 2
N = 400 # 3 per class (ignoring 3rd image in each class, done in next cell)

In [3]:
faces.shape

(24, 21, 600)

   ### Split Data

In [4]:

#select images 2 and 3 from each subject for training
faces_flat = faces.reshape(-1,faces.shape[2])
#ignore image 3 (illumination) of all classes (creates a dataset of 400 samples)
idx = []
for n in range(600):
    if n%3 == 0 or n%3 == 1:
        idx = idx + [n]
faces_flat = faces_flat[:,idx]
train_idx =[]
test_idx = []
print(faces_flat.shape)
#create training set (75% = 300)
training_faces = faces_flat[:,:300]
test_faces = faces_flat[:,300:400]

# #rn test and training set have neutral face and expression alternating
#Create label Arrays
training_labels = [-1,1]*150
test_labels = [-1,1]*50

# training_faces = np.vstack(([-1,1]*150, training_faces))
# test_faces = np.vstack(([-1,1]*50, test_faces))

#datasets have alternating neutral and expression pictures
#rearrange saw that all the neutral come first then expression
training_faces = np.hstack((training_faces[:,::2], training_faces[:,1::2])) #0 -149 neutral, 150 - 299 expression
test_faces = np.hstack((test_faces[:,::2], test_faces[:,1::2])) #0-49 neutral, 50 - 99 expression
print(training_faces.shape, test_faces.shape)

(504, 400)
(504, 300) (504, 100)


### Perform PCA

In [36]:
#reduce dimensions from 504 to 1 (only 2 classes, so 1 dimension is the only possibility).
A, train_mean = PCA(training_faces, 10)
X_PCA = A @ (training_faces - train_mean)
X_test_PCA = A @ (test_faces - train_mean)
print(X_PCA.shape)


(10, 300)


### Perform MDA

In [75]:
#We have two classes here
A,class_means,sigma_w = MDA(X_PCA, 1, 2)
X_MDA = A @ X_PCA
X_test_MDA = A @ X_test_PCA

X_train = X_PCA.real
X_test = X_test_PCA.real
y_train = np.concatenate((-np.ones((1,150)), np.ones((1,150))), axis = 1)
y_test = np.concatenate((-np.ones((1,50)), np.ones((1,50))), axis = 1)


### Defining stump as a class  

In [38]:
class DecisionStump:
    def __init__(self):
        self.k = None#idx
        self.theta = None#threshold
        self.pol = 1#polarity

    def predict(self, X):
        return np.sign(self.pol * (X[self.k,:] - self.theta)).reshape(1,-1)


### Train AdaBoost


In [105]:
l, N_train = X_train.shape

T = 500 #no of rounds


#T stumps for adaboost
stumps = np.array([DecisionStump() for t in range(T)], dtype=object)

err = np.zeros(T)

a = np.zeros(T) #parameters (vote weightage of each weak learner)

#initialize weight w for priors
w = np.ones((1,N_train))

for t in range(T):
    #priors
    P = w/np.sum(w) # N_train x 1
    #returns feature index for best stump (minimum error weighted by w), theta for threshold and error value err and polarity
    stumps[t],err[t] = best_stump(X_train, y_train, P)
    #prevent divide by 0
    e = max(err[t],1e-5)
    a[t] = 0.5 * np.log((1-e)/e)
    #update weights
    w *= np.exp(- a[t] * y_train * stumps[t].predict(X_train))

### Training and Testing error

In [106]:
final = np.zeros((1,N_train))
for t in range(T):
    final += a[t] * stumps[t].predict(X_train)
y_train_predicted = np.sign(final)
train_error = np.mean(y_train != y_train_predicted)

In [107]:
print(train_error)

0.0


In [108]:
final = np.zeros((1,X_test.shape[1]))
for t in range(T):
    final += a[t] * stumps[t].predict(X_test)
y_test_predicted = np.sign(final)
test_error = np.mean(y_test != y_test_predicted)
print(test_error)

0.15
